# Generate Processing Configuration Suite

This notebook generates a set of configuration files for the EMIMesh pipeline to test the impact of different image processing and meshing parameters on the final result.

## Objective
We want to evaluate how variations in resolution, morphological operation strength, and smoothing affect the resulting tetrahedral meshes of brain tissue.

## Configuration Design

The generated files follow these constraints:
- **Fixed Operations**: All files include `removeislands`, `dilate`, `smooth`, and `erode`.
- **Fixed removeislands**: `minsize=10000`.
- **Symmetry**: `dilate radius` == `erode radius` $\in [0, 4]$.
- **Smoothing**: `iterations` $\in [1, 4]$, `radius` $\in [1, 4]$.
- **Resolution**: `mip` $\in [0, 4]$, `dx` $\in [20, 200]$.
- **Envelop Size**: 20% to 50% of `dx`.

### Parameter Groups

#### Group A: Resolution & Scale (Constants: dil=2, ero=2, sm_it=1, sm_rad=2)
| mip | dx | env | Name Suffix |
| :--- | :--- | :--- | :--- |
| 0 | 20 | 4 | `mip0_dx20_env4` |
| 2 | 50 | 10 | `mip2_dx50_env10` |
| 4 | 100 | 20 | `mip4_dx100_env20` |
| 4 | 100 | 50 | `mip4_dx100_env50` |
| 4 | 200 | 40 | `mip4_dx200_env40` |
| 4 | 200 | 100 | `mip4_dx200_env100` |

#### Group B: Morphological Radius (Constants: mip=4, dx=100, env=50, sm_it=1, sm_rad=2)
| dil/ero radius | Name Suffix |
| :--- | :--- |
| 0 | `mip4_dx100_env50_rad0` |
| 1 | `mip4_dx100_env50_rad1` |
| 2 | `mip4_dx100_env50_rad2` |
| 3 | `mip4_dx100_env50_rad3` |
| 4 | `mip4_dx100_env50_rad4` |

#### Group C: Smoothing Strength (Constants: mip=4, dx=100, env=50, dil=2, ero=2)
| iterations | radius | Name Suffix |
| :--- | :--- | :--- |
| 1 | 1 | `mip4_dx100_env50_sm11` |
| 1 | 4 | `mip4_dx100_env50_sm14` |
| 4 | 1 | `mip4_dx100_env50_sm41` |
| 4 | 4 | `mip4_dx100_env50_sm44` |

In [ ]:
import yaml
import os
from pathlib import Path

# Force double quotes for all strings in YAML output
def string_representer(dumper, data):
    return dumper.represent_scalar('tag:yaml.org,2002:str', data, style='"')

yaml.add_representer(str, string_representer)

def create_config(
    output_dir, 
    mip, size, dx, env, dil_ero_rad, sm_it, sm_rad, rm_island_minsize,
    idx=0,
    output_name_conv=0, 
    cell_type="neuron", 
    neuron_type="4P", 
    output_folder="cells",
    surronding_cells=0,
    cell_point_type=None
):
    # Construct the name based on project conventions
    if output_name_conv == 0:
        internal_name = f"processing/mip{mip}_dx{dx}_rmis10000_dil{dil_ero_rad}_sm{sm_it}{sm_rad}_er{dil_ero_rad}_env{env}"
    elif output_name_conv == 1:
        if cell_type == "neuron":
            internal_name = f"{output_folder}/{cell_type}s/neuron_{neuron_type}_{idx}"
        else:
            internal_name = f"{output_folder}/{cell_type}s/{cell_type}_{idx}"
    elif output_name_conv == 2:
            internal_name = f"{output_folder}/{cell_point_type}/size{size}_ncells{surronding_cells}_{idx}"
    
    config = {
        "name": internal_name,
        "raw": {
            "cloudpath": "precomputed://gs://iarpa_microns/minnie/minnie65/seg_m1300",
            "position": "0-0-0",
            "mip": mip,
            "size": size,
            "cell_type": cell_type,
            "cell_idx": idx,
            "cell_padding": 20,
            "cell_table_name": "aibs_metamodel_celltypes_v661",
            "cell_keep_surrounding": surronding_cells > 0,
        },
        "processing": {
            "dx": dx,
            "operation": [
                f"removeislands minsize={rm_island_minsize}",
                f"dilate radius={dil_ero_rad}",
                f"smooth iterations={sm_it} radius={sm_rad}",
                f"erode radius={dil_ero_rad}"
            ]
        },
        "meshing": {
            "envelopsize": env
        }
    }

    # Add cell type if neuron
    if cell_type == "neuron":
        config["raw"]["cell_neuron_type"] = neuron_type

    # Keep surrounding cells if specified
    if surronding_cells > 0:
        config["processing"]["operation"].insert(0, f"ncells ncells={surronding_cells}")

    # Add cell point type if specified
    if cell_point_type is not None:
        config["raw"]["cell_point_type"] = cell_point_type
    
    if output_name_conv == 2:
        file_path = output_dir / f"{cell_point_type}_size{size}_ncells{surronding_cells}_{idx}.yml"
    elif cell_type == "neuron":
        file_path = output_dir / f"{cell_type}_{neuron_type}_{idx}.yml"
    else:
        file_path = output_dir / f"{cell_type}_{idx}.yml"
    
    
    with open(file_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    return file_path

## Test different processing parameters

In [ ]:
output_dir = Path("../emimesh/config_files/single_neuron_processing")
output_dir.mkdir(parents=True, exist_ok=True)

# Parameter sets
configs_to_generate = []

size = 50_000

# Group A: Resolution & Scale (dil=2, ero=2, sm_it=1, sm_rad=2)
group_a = [
    (0, size, 20, 4, "mip0_dx20_env4"),
    (2, size, 50, 10, "mip2_dx50_env10"),
    (4, size, 100, 20, "mip4_dx100_env20"),
    (4, size, 100, 50, "mip4_dx100_env50"),
    (4, size, 200, 40, "mip4_dx200_env40"),
    (4, size, 200, 100, "mip4_dx200_env100"),
]
for mip, size, dx, env, suffix in group_a:
    configs_to_generate.append((mip, size, dx, env, 2, 1, 2, 10_000, suffix))

# Group B: Morphological Radius (mip=4, dx=100, env=50, sm_it=1, sm_rad=2)
for rad in range(5):
    configs_to_generate.append((4, size, 100, 50, rad, 1, 2, 10_000, f"mip4_dx100_env50_rad{rad}"))

# Group C: Smoothing Strength (mip=4, dx=100, env=50, dil=2, ero=2)
smooth_variations = [
    (1, 1, "sm11"),
    (1, 4, "sm14"),
    (4, 1, "sm41"),
    (4, 4, "sm44"),
]
for it, rad, suffix_part in smooth_variations:
    configs_to_generate.append((4, size, 100, 50, 2, it, rad, 10_000, f"mip4_dx100_env50_{suffix_part}"))

# Generate files
generated_files = []
for params in configs_to_generate:
    path = create_config(output_dir, *params)
    generated_files.append(path)

print(f"Successfully generated {len(generated_files)} configuration files in {output_dir}")
for f in generated_files:
    print(f)

## Generate fixed config for many neurons

In [ ]:
### Low resolution dataset
output_folder = "cells_lowres"
output_dir = Path(f"../emimesh/config_files/{output_folder}")
output_dir.mkdir(parents=True, exist_ok=True)

# https://tutorial.microns-explorer.org/annotation-tables.html or emimesh/src/emimesh/cave_query.py
neuron_types = ["23P","4P","6P-IT","5P-IT","6P-CT","BC","MC","5P-ET","BPC","5P-NP"]
cell_types = ["astrocyte","microglia"] # Skip pericytes, OPCs, oligos

N = 50
idx_start=100

for neuron_type in neuron_types:
    for idx in range(idx_start, idx_start + N):
        create_config(output_dir, 4, 300_000, 200, 100, 2, 1, 2, 10_000, idx, 1, cell_type="neuron", neuron_type=neuron_type, output_folder=output_folder)

for cell_type in cell_types:
    for idx in range(idx_start, idx_start + N):
        create_config(output_dir, 4, 300_000, 200, 100, 2, 1, 2, 10_000, idx, 1, cell_type=cell_type, neuron_type=None, output_folder=output_folder)
        
# Print n files created
print(f"Created {N*len(neuron_types) + N*len(cell_types)} config files")

In [ ]:
### High resolution dataset
output_folder = "cells_highres"
output_dir = Path(f"../emimesh/config_files/{output_folder}")
output_dir.mkdir(parents=True, exist_ok=True)

neuron_types = ["23P","4P","6P-IT","5P-IT","6P-CT","BC","MC","5P-ET","BPC","5P-NP"]
cell_types = ["microglia"] # Skip astrocytes, pericytes, OPCs, oligos

N = 20

for neuron_type in neuron_types:
    for idx in range(N):
        create_config(output_dir, 2, 300_000, 50, 20, 2, 1, 2, 10_000, idx, 1, cell_type="neuron", neuron_type=neuron_type, output_folder=output_folder)

for cell_type in cell_types:
    for idx in range(N):
        create_config(output_dir, 2, 300_000, 50, 20, 2, 1, 2, 10_000, idx, 1, cell_type=cell_type, neuron_type=None, output_folder=output_folder)
        
# Print n files created
print(f"Created {N*len(neuron_types) + N*len(cell_types)} config files")

## Generate cubes

In [ ]:
### Low resolution dataset
output_folder = "cells_cube_lowres"
output_dir = Path(f"../emimesh/config_files/{output_folder}")
output_dir.mkdir(parents=True, exist_ok=True)

N = 20
idx_start=0

cell_point_types = ["soma", "end", "branch", "segment"]
surronding_cells = [5, 10, 20]

i = 0
for surr in surronding_cells:
    for cell_point_type in cell_point_types:
        for idx in range(idx_start, idx_start + N):
            create_config(
                output_dir, 
                4, 30_000, 200, 100, 2, 1, 2, 1_000, # mip, size, dx, env, dil_ero_rad, sm_it, sm_rad, rm_island_minsize,
                idx, 
                2, 
                cell_type="neuron", 
                neuron_type="4P", 
                output_folder=output_folder,
                surronding_cells=surr,
                cell_point_type=cell_point_type
            )
            i += 1
        
# Print n files created
print(f"Created {i} config files")

## Next
Run EMIMesh pipeline with
```bash
cd emimesh
conda activate snakemake
snakemake --configfile config_files/single_neuron_processing/neuron_mip0_dx20_env4.yml --use-conda --cores 8
```

If running multiple at the same time, create the environment first.
```bash
snakemake --configfile config_files/single_neuron_processing/neuron_mip0_dx20_env4.yml --use-conda --cores 1 --conda-create-envs-only
```
 